In [ ]:
# ============================ CONFIG ============================
DATA_ROOT = "SisFall_Dataset"
OUT_DIR   = "/path"
CACHE_DIR = "/path/cache"

EXCLUDE_ADL = {"D14", "D15", "D16"}      
FS, WIN, STEP = 200, 300, 150
WIN_BEFORE, WIN_AFTER, EXTRA_OFFSET = 200, 100, 300
LOWPASS_HZ, BUTTER_ORDER = 20, 5
SINDY_DEGREE, SINDY_DT = 3, 1.0 / 200

EPOCHS, BATCH, LR = 100, 16, 1e-4
LYAP_ALPHA, M_INIT, M_ANCHOR = 0.1, 1e-2, 0.5
SEED = 42

QUICK_TEST = False
MAX_FILES = None
if QUICK_TEST:
    EPOCHS, MAX_FILES = 3, 60

: 

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
import matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import (Input, Dense, Dropout, BatchNormalization,
                                     Concatenate, Conv1D, GlobalAveragePooling1D,
                                     GlobalMaxPool1D, Reshape, Lambda)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
import pysindy as ps
from tcn import TCN
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **k): return x

os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(CACHE_DIR, exist_ok=True)
print("TF", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))
for g in tf.config.list_physical_devices("GPU"):
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass

COLUMNS = ['ADXL345_x','ADXL345_y','ADXL345_z','ITG3200_x','ITG3200_y','ITG3200_z',
           'MMA8451Q_x','MMA8451Q_y','MMA8451Q_z']
SCALE = [0.003906250]*3 + [0.061035156]*3 + [0.000976563]*3
try:
    _ = ps.SINDyDerivative(kind="savitzky_golay", left=2, right=0, order=3); _HAS_SG = True
except Exception:
    _HAS_SG = False

In [ ]:
# ============================ PREPROCESSING ============================
def butter_lowpass_filter(data, cutoff=LOWPASS_HZ, fs=FS, order=BUTTER_ORDER):
    b, a = butter(order, cutoff / (0.5 * fs), btype='low', analog=False)
    return filtfilt(b, a, data)

def load_and_preprocess_file(file_path):
    rows = []
    with open(file_path) as f:
        for line in f:
            v = line.replace(',', ' ').replace(';', ' ').strip().split()
            if len(v) == 9:
                rows.append([float(x) for x in v])
    if not rows:
        return None
    df = pd.DataFrame(rows, columns=COLUMNS)
    for c, s in zip(COLUMNS, SCALE):
        df[c] *= s
    df[COLUMNS] = StandardScaler().fit_transform(df[COLUMNS])   
    for col in COLUMNS:
        df[col] = butter_lowpass_filter(df[col].values)
    return df.values

def fit_sindy(window):
    kw = dict(optimizer=ps.STLSQ(threshold=0.1), feature_library=ps.PolynomialLibrary(degree=SINDY_DEGREE),
              differentiation_method=(ps.SINDyDerivative(kind="savitzky_golay", left=2, right=0, order=3)
                                      if _HAS_SG else ps.SmoothedFiniteDifference()))
    return np.array([ps.SINDy(**kw).fit(window[:, i:i+3], t=SINDY_DT).coefficients().flatten()
                     for i in range(0, 9, 3)], dtype=np.float32)        # (3, 60)

def process_fall_files(dataset_dir):
    W, Y, S, C = [], [], [], []
    files = [os.path.join(d, fn) for d, _, fl in os.walk(dataset_dir)
             for fn in fl if fn.startswith('F') and fn.endswith('.txt')]
    if MAX_FILES: files = files[:MAX_FILES]
    for fp in tqdm(files, desc="Fall files"):
        code = os.path.basename(fp).split('_')[0]
        data = load_and_preprocess_file(fp)
        if data is None: continue
        mag = np.sqrt((data[:, 0:3] ** 2).sum(1))
        peaks, _ = find_peaks(mag, height=np.max(mag) * 0.8)
        if len(peaks) == 0: continue
        pk = peaks[np.argmax(mag[peaks])]
        s0 = max(0, pk - WIN_BEFORE); e0 = min(len(data), pk + WIN_AFTER)
        if e0 - s0 == WIN:
            W.append(data[s0:e0]); Y.append(1); S.append(fit_sindy(data[s0:e0])); C.append(code)
        es = max(0, s0 - EXTRA_OFFSET); ee = es + WIN
        if ee <= len(data):
            W.append(data[es:ee]); Y.append(1); S.append(fit_sindy(data[es:ee])); C.append(code)
    return W, Y, S, C

def process_adl_files(dataset_dir, max_per_type):
    W, Y, S, C = [], [], [], []
    files = [os.path.join(d, fn) for d, _, fl in os.walk(dataset_dir)
             for fn in fl if fn.startswith('D') and fn.endswith('.txt')
             and fn.split('_')[0] not in EXCLUDE_ADL]
    if MAX_FILES: files = files[:MAX_FILES]
    counts = {}
    for fp in tqdm(files, desc="ADL files"):
        code = os.path.basename(fp).split('_')[0]
        counts.setdefault(code, 0)
        if max_per_type and counts[code] >= max_per_type: continue
        data = load_and_preprocess_file(fp)
        if data is None: continue
        for start in range(0, len(data) - WIN + 1, STEP):
            W.append(data[start:start + WIN]); Y.append(0)
            S.append(fit_sindy(data[start:start + WIN])); C.append(code); counts[code] += 1
    return W, Y, S, C

def build_dataset():
    cache = os.path.join(CACHE_DIR, f"sisfall_main_deg{SINDY_DEGREE}_q{int(QUICK_TEST)}.npz")
    if os.path.exists(cache):
        print("loading cache", cache)
        d = np.load(cache, allow_pickle=True)
        return d["X"], d["S"], d["y"], d["codes"]
    fW, fY, fS, fC = process_fall_files(DATA_ROOT)
    n_types = max(1, len({fn.split('_')[0] for d, _, fl in os.walk(DATA_ROOT) for fn in fl
                          if fn.startswith('D') and fn.endswith('.txt') and fn.split('_')[0] not in EXCLUDE_ADL}))
    max_per_type = max(1, len(fY) // n_types)
    aW, aY, aS, aC = process_adl_files(DATA_ROOT, max_per_type)
    X = np.array(fW + aW, np.float32); S = np.array(fS + aS, np.float32)
    y = np.array(fY + aY, np.int64);   codes = np.array(fC + aC)
    np.savez_compressed(cache, X=X, S=S, y=y, codes=codes)
    print(f"falls={int((y==1).sum())} adls={int((y==0).sum())} ({n_types} ADL types) | X{X.shape} S{S.shape}")
    return X, S, y, codes

def make_split(y, seed):
    idx = np.arange(len(y))
    tr, tmp = train_test_split(idx, test_size=0.2, random_state=seed, stratify=y)
    va, te = train_test_split(tmp, test_size=0.5, random_state=seed, stratify=y[tmp])
    return tr, va, te

In [ ]:
# ============================ MODEL ============================
def build_ltcnet(win_shape, sindy_shape):
    sig = Input(win_shape, name="signal")
    x = BatchNormalization()(sig)
    x = TCN(nb_filters=64, kernel_size=3, dilations=[1, 2, 4, 8, 16],
            return_sequences=True, dropout_rate=0.1)(x)
    tcn = GlobalAveragePooling1D()(x)                               
    coef = Input(sindy_shape, name="sindy")                         
    branches = []
    for i in range(3):
        b = Lambda(lambda z, i=i: z[:, i, :])(coef)                  
        b = Reshape((sindy_shape[1], 1))(b)
        b = Conv1D(16, 3, activation="relu")(b)
        b = GlobalMaxPool1D()(b)                                  
        branches.append(b)
    s = Concatenate()(branches)                                  
    s = Dense(64, activation="relu")(s)                            
    f = Concatenate()([tcn, s])                                    
    f = BatchNormalization()(f)
    f = Dropout(0.3)(f)
    f = Dense(64, activation="relu")(f)
    out = Dense(2, activation="softmax")(f)
    return Model([sig, coef], out)

class FDModelLyap(Model):
    """LTC-Net core + trainable-threshold hinge Lyapunov:
       V_i = ||y_i - yhat_i||^2 ;  L = CE + alpha*( mean ReLU(V_i - m) + m_anchor*m )."""
    def __init__(self, core, alpha=LYAP_ALPHA, m_init=M_INIT, m_anchor=M_ANCHOR):
        super().__init__()
        self.core, self.alpha, self.m_anchor = core, alpha, m_anchor
        self.m = self.add_weight(name="lyap_m", shape=(), trainable=True,
                                 initializer=tf.keras.initializers.Constant(m_init))
    def call(self, x, training=False): return self.core(x, training=training)
    def train_step(self, data):
        x, y = data
        with tf.GradientTape() as tape:
            yp = self.core(x, training=True)
            ce = tf.reduce_mean(tf.keras.losses.categorical_crossentropy(y, yp))
            V = tf.reduce_sum(tf.square(y - yp), axis=1)              
            lyap = tf.reduce_mean(tf.nn.relu(V - self.m)) + self.m_anchor * self.m
            loss = ce + self.alpha * lyap
        v = self.trainable_variables
        self.optimizer.apply_gradients(zip(tape.gradient(loss, v), v))
        return {"loss": loss, "ce": ce, "m": self.m}
    def test_step(self, data):
        x, y = data
        ce = tf.reduce_mean(tf.keras.losses.categorical_crossentropy(y, self.core(x, training=False)))
        return {"loss": ce}

In [ ]:
# ============================ TRAIN ============================
X, S, y, codes = build_dataset()
tr, va, te = make_split(y, SEED)
print(f"train {len(tr)} | val {len(va)} | test {len(te)} | "
      f"test falls {int((y[te]==1).sum())} / ADL {int((y[te]==0).sum())}")

tf.keras.utils.set_random_seed(SEED)
core = build_ltcnet(X.shape[1:], S.shape[1:])
print(f"parameters: {core.count_params():,}")
model = FDModelLyap(core, alpha=LYAP_ALPHA)
model.compile(optimizer=Adam(LR))
hist = model.fit([X[tr], S[tr]], to_categorical(y[tr], 2),
                 validation_data=([X[va], S[va]], to_categorical(y[va], 2)),
                 epochs=EPOCHS, batch_size=BATCH, verbose=1)

In [ ]:
# ============================ EVALUATE ============================
pred = model.predict([X[te], S[te]], verbose=0).argmax(1)
t = y[te]
res = dict(
    accuracy=round(accuracy_score(t, pred) * 100, 2),
    precision=round(precision_score(t, pred, average="weighted", zero_division=0) * 100, 2),
    recall=round(recall_score(t, pred, average="weighted", zero_division=0) * 100, 2),
    f1=round(f1_score(t, pred, average="weighted", zero_division=0) * 100, 2),
    f1_macro=round(f1_score(t, pred, average="macro", zero_division=0) * 100, 2),
    f1_fall=round(f1_score(t, pred, pos_label=1, average="binary", zero_division=0) * 100, 2),
    learned_m=round(float(model.m.numpy()), 4),
)
print("\nLTC-Net (SisFall) — test metrics")
for k, v in res.items(): print(f"  {k:11s}: {v}")
print("\n", classification_report(t, pred, target_names=["ADL", "Fall"], digits=4))
with open(os.path.join(OUT_DIR, "ltcnet_sisfall_metrics.json"), "w") as f:
    json.dump(res, f, indent=2)

cm = confusion_matrix(t, pred, labels=[0, 1])
plt.figure(figsize=(3.4, 3.1)); plt.imshow(cm, cmap="Blues")
plt.title("LTC-Net — SisFall")
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.xticks([0, 1], ["ADL", "Fall"]); plt.yticks([0, 1], ["ADL", "Fall"])
plt.xlabel("Predicted"); plt.ylabel("True"); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "ltcnet_sisfall_cm.png"), dpi=130); plt.close()
print("saved metrics + confusion matrix to", OUT_DIR)